In [ ]:
import numpy as np
from PIL import Image
from datasets import load_dataset, get_dataset_config_names
import random
import matplotlib.pyplot as plt

# =============================================================================
# ✨ 데이터셋 소개: ChartQA (차트 QA) ✨
# 이 데이터셋은 '차트' 이미지를 보여주고, 그 차트에 대해 질문(query)하고,
# 정답(label)을 찾는 데이터셋입니다. 🤖
# 마치 AI가 그래프를 보고 질문에 답하는 능력을 훈련하는 것과 같아요!
# 🎯 목표: 이미지를 이해하고, 텍스트 질문에 답하는 AI 시스템을 구축하는 것입니다.
# =============================================================================

DATASET_NAME = "HuggingFaceM4/ChartQA"
SAMPLE_COUNT = 10 # 테스트를 위해 적은 수의 샘플만 사용합니다.

print(f"🌟 튜터가 준비한 데이터셋: {DATASET_NAME} (차트 QA)")

# 1. 데이터셋 Config 확인 (필수 단계!)
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    selected_config = configs[0]
except Exception as e:
    print("ℹ️ 해당 데이터셋은 별도의 Config가 없거나 기본(default) 설정만 제공됩니다.")
    selected_config = None

# 2. 데이터셋 로드 전략: 스트리밍 시도 및 대체 로드
dataset = None
print("-" * 70)

try:
    # 🚀 1단계: 스트리밍(Streaming) 모드로 로드를 시도합니다. (메모리 효율성 최고!)
    print("💡 (Step 1/3) 스트리밍 모드를 사용하여 학습 데이터셋을 로드합니다...")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✨ 성공! 데이터셋을 스트리밍 모드로 로드했습니다. (매우 빠름!)")

except Exception as e:
    # 🚨 스트리밍 로드 실패 시 예외 처리 (필요한 경우에만 일부를 다운로드)
    print(f"⚠️ 스트리밍 로드에 실패했습니다. 오류: {e}")
    print("💡 대체 전략: 소수의 테스트 데이터만 일반 모드로 로드하여 진행합니다.")
    try:
        # 테스트 세트는 크기가 작으므로 일반 모드로 로드해도 무방합니다.
        dataset = load_dataset(DATASET_NAME, split='test')
        print("✨ 성공! 테스트 데이터셋 일부를 일반 모드로 로드했습니다.")
    except Exception as e2:
        print(f"❌ 데이터셋 로드 자체에 실패했습니다. ({e2}) 코드를 종료합니다.")
        exit()

# 3. 샘플링 및 반복자 준비 (핵심 패턴!)
# 🌟 주의: 스트리밍 데이터셋은 len()을 쓸 수 없습니다. .take()와 iter()를 사용합니다.
print("-" * 70)
print(f"✨ (Step 2/3) 상위 {SAMPLE_COUNT}개의 샘플을 선택하여 실습을 시작합니다.")

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)인 경우
    print("✅ 스트리밍 Iterator를 사용합니다.")
    # take()를 사용해 상위 N개를 자르고, list()로 변환하여 사용 가능하게 만듭니다.
    sampled_dataset_iterator = iter(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset)인 경우
    print("✅ 일반 Dataset을 사용합니다.")
    # .select()가 아닌, take() 방식과 유사하게 list()로 변환하여 사용합니다.
    sampled_dataset = list(dataset.take(SAMPLE_COUNT))
    # list로 변환했으므로, 직접 반복 가능한 객체를 만듭니다.
    sampled_dataset_iterator = iter(sampled_dataset)

# 4. 실습 함수 정의 (튜터 모드: 친절한 설명 추가)
def analyze_chartqa_sample(sample: dict, sample_index: int):
    """
    하나의 샘플을 받아 분석 결과를 예쁘게 보여주는 함수입니다.
    (우리가 AI처럼 데이터를 '해석'하는 시뮬레이션 과정입니다!)
    """
    # 🖼️ 이미지 분석: 차트를 시각화합니다. (학습의 핵심!)
    image = sample['image']
    plt.figure(figsize=(6, 3))
    plt.imshow(np.array(image))
    plt.title(f"Chart Data (Sample {sample_index})")
    plt.axis('off')
    plt.show()
    
    # ❓ 질문 분석: AI가 어떤 질문을 받는지 확인합니다.
    query = sample['query']
    print(f"\n📝 [질문 (Query)]: {query}")
    
    # 🎯 분류 분석: 질문과 차트를 바탕으로 AI가 무엇을 판단했는지 확인합니다.
    human_or_machine_label = sample['human_or_machine']
    
    # 레이블을 예쁘게 출력하기 위해 튜플로 변환합니다.
    human_str = "🧑 Human"
    machine_str = "🤖 Machine"
    
    if human_or_machine_label == 'human':
        print(f"   -> 🧠 분석 결과: {human_str} 판단 (사람이 만든 듯한 패턴!)")
    elif human_or_machine_label == 'machine':
        print(f"   -> 🤖 분석 결과: {machine_str} 판단 (기계적이고 규칙적인 패턴!)")
    else:
        print("   -> ❓ 분석 결과: 알 수 없는 분류입니다.")

# 5. 실습 실행 (반복문)
print("\n" + "#" * 70)
print("🎉 (Step 3/3) AI 분석 실습 시작! (이 코드가 실행되는 과정이 바로 AI의 사고 과정입니다!)")
print("#" * 70)

# 반복 가능한 Iterator를 사용하여 샘플을 가져옵니다.
for i, sample_data in enumerate(sampled_dataset_iterator):
    print(f"\n{'='*20} ✨ 샘플 {i+1} 분석 시작 ✨ {'='*20}")
    
    # 데이터를 로드받는 과정에서 NumPy 배열로 명시적으로 처리합니다.
    # (PIL Image 객체를 바로 plt.imshow()에 넣기 위해 배열로 변환합니다.)
    sample_data['image'] = np.array(sample_data['image']) 
    
    # 우리의 분석 함수를 호출합니다.
    analyze_chartqa_sample(sample_data, i + 1)
    
    # 학습자가 너무 많은 창을 보지 않도록, 3개의 샘플만 보고 중단합니다.
    if i >= 2:
        break

print("\n" + "#" * 70)
print("🎉👏🎉 모든 실습을 성공적으로 완료했습니다! 👏🎉🎉")
print("👏 팁: 이처럼 여러 종류의 특징(Image, String, Class)을 한 번에 다루는 것이 바로 AI의 핵심 능력입니다! 👏👏")